# Used Car Price Prediction

## Business context
Rusty Bargain, a used-car sales service, is developing an application to estimate a vehicle’s market value. The goal is to predict vehicle prices from historical listing information, while considering prediction accuracy, training speed, and inference speed. The target is **`Price`**, measured in euros.

This notebook explores the data, cleans inconsistent records, compares Linear Regression, Random Forest, and LightGBM, and evaluates the selected model on a held-out test set.

## 1. Data preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv('../data/car_data.csv')

print('Dataset shape:')
print(data.shape)

print('\nFirst rows:')
display(data.head())

print('\nDataset information:')
data.info()

print('\nDescriptive statistics:')
display(data.describe())

print('\nMissing values by column:')
print(data.isna().sum())

print('\nDuplicate records:')
print(data.duplicated().sum())

Dimensões do dataset:
(354369, 16)

Primeiras linhas:


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17



Informações do dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object


,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000



Valores ausentes por coluna:
DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64

Número de registros duplicados:
262


The original dataset contains **354,369 records and 16 columns**, including the target `Price`. Initial checks identify missing values, duplicate records, and potentially inconsistent entries. These issues are examined before modeling.

In [2]:
# Inspect inconsistent values

print('Zero prices:')
print((data['Price'] == 0).sum())

print('\nRegistration years before 1900:')
print((data['RegistrationYear'] < 1900).sum())

print('\nRegistration years after 2016:')
print((data['RegistrationYear'] > 2016).sum())

print('\nZero power:')
print((data['Power'] == 0).sum())

print('\nPower above 500:')
print((data['Power'] > 500).sum())

print('\nPrice distribution:')
display(data['Price'].describe())

print('\nRegistration year distribution:')
display(data['RegistrationYear'].value_counts().sort_index())

Preços iguais a zero:
10772

Anos de registro anteriores a 1900:
66

Anos de registro posteriores a 2016:
14530

Potência igual a zero:
40225

Potência superior a 500:
459

Distribuição dos preços:


count    354369.000000
mean       4416.656776
std        4514.158514
min           0.000000
25%        1050.000000
50%        2700.000000
75%        6400.000000
max       20000.000000
Name: Price, dtype: float64


Distribuição dos anos de registro:


1000    37
1001     1
1039     1
1111     3
1200     1
        ..
9000     3
9229     1
9450     1
9996     1
9999    26
Name: RegistrationYear, Length: 151, dtype: int64

Data quality checks identified **10,772 zero-price records**, **66 registration years before 1900**, **14,530 registration years after 2016**, and implausible values in `Power`. The next step removes invalid records, replaces invalid power values with the median, fills missing categorical values, and drops administrative columns.

In [3]:
# Remove duplicate records
data = data.drop_duplicates()

# Remove zero-price records
data = data[data['Price'] > 0].copy()

# Remove inconsistent registration years
data = data[
    (data['RegistrationYear'] >= 1900) &
    (data['RegistrationYear'] <= 2016)
].copy()

# Replace invalid power values
data.loc[
    (data['Power'] == 0) | (data['Power'] > 500),
    'Power'
] = np.nan

data['Power'] = data['Power'].fillna(data['Power'].median())

# Fill missing categorical values
categorical_columns = [
    'VehicleType',
    'Gearbox',
    'Model',
    'FuelType',
    'NotRepaired'
]

data[categorical_columns] = data[categorical_columns].fillna('unknown')

# Remove administrative columns
columns_to_drop = [
    'DateCrawled',
    'DateCreated',
    'LastSeen',
    'NumberOfPictures',
    'PostalCode'
]

data = data.drop(columns_to_drop, axis=1)

# Check cleaning results
print('Shape after cleaning:')
print(data.shape)

print('\nRemaining missing values:')
print(data.isna().sum())

print('\nRemaining duplicate records:')
print(data.duplicated().sum())

display(data.head())

Dimensões após o tratamento:
(329506, 11)

Valores ausentes restantes:
Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Mileage              0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
dtype: int64

Registros duplicados restantes:
25269


,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired
0,480,unknown,1993,manual,110.0,golf,150000,0,petrol,volkswagen,unknown
1,18300,coupe,2011,manual,190.0,unknown,125000,5,gasoline,audi,yes
2,9800,suv,2004,auto,163.0,grand,125000,8,gasoline,jeep,unknown
3,1500,small,2001,manual,75.0,golf,150000,6,petrol,volkswagen,no
4,3600,small,2008,manual,69.0,fabia,90000,7,gasoline,skoda,no


After cleaning, the dataset contains **329,506 records and 11 columns**. Duplicate records, zero prices, and invalid registration years were removed. Invalid power values were imputed with the median, and missing categorical values were assigned an `unknown` category.

In [4]:
# Split data before encoding

from sklearn.model_selection import train_test_split

features = data.drop('Price', axis=1)
target = data['Price']

features_train, features_valid_test, target_train, target_valid_test = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=12345
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_valid_test,
    target_valid_test,
    test_size=0.5,
    random_state=12345
)

print('Training:', features_train.shape)
print('Validation:', features_valid.shape)
print('Test:', features_test.shape)

Treinamento: (197703, 10)
Validação: (65901, 10)
Teste: (65902, 10)


In [5]:
# Apply one-hot encoding
features_train_ohe = pd.get_dummies(features_train, drop_first=True)
features_valid_ohe = pd.get_dummies(features_valid, drop_first=True)
features_test_ohe = pd.get_dummies(features_test, drop_first=True)

# Align validation and test columns with training columns
features_valid_ohe = features_valid_ohe.reindex(
    columns=features_train_ohe.columns,
    fill_value=0
)

features_test_ohe = features_test_ohe.reindex(
    columns=features_train_ohe.columns,
    fill_value=0
)

print('Training:', features_train_ohe.shape)
print('Validation:', features_valid_ohe.shape)
print('Test:', features_test_ohe.shape)

print('\nMissing values:')
print(features_train_ohe.isna().sum().sum())
print(features_valid_ohe.isna().sum().sum())
print(features_test_ohe.isna().sum().sum())

Treinamento: (197703, 312)
Validação: (65901, 312)
Teste: (65902, 312)

Valores ausentes:
0
0
0


In [6]:
# Scale numeric features using training statistics

from sklearn.preprocessing import StandardScaler

numeric = [
    'RegistrationYear',
    'Power',
    'Mileage',
    'RegistrationMonth'
]

features_train_ohe = features_train_ohe.copy()
features_valid_ohe = features_valid_ohe.copy()
features_test_ohe = features_test_ohe.copy()

scaler = StandardScaler()
scaler.fit(features_train_ohe[numeric])

features_train_ohe[numeric] = scaler.transform(features_train_ohe[numeric])
features_valid_ohe[numeric] = scaler.transform(features_valid_ohe[numeric])
features_test_ohe[numeric] = scaler.transform(features_test_ohe[numeric])

print('Training:', features_train_ohe.shape)
print('Validation:', features_valid_ohe.shape)
print('Test:', features_test_ohe.shape)

display(features_train_ohe[numeric].head())

Treinamento: (197703, 312)
Validação: (65901, 312)
Teste: (65902, 312)


,RegistrationYear,Power,Mileage,RegistrationMonth
265475,-0.375447,0.990728,0.580808,0.593237
288186,0.633961,-0.284809,0.580808,0.593237
11170,-0.808050,-0.873518,-1.018434,-0.222122
349104,0.201358,1.128093,0.580808,1.680381
339603,-0.519648,-0.186691,-2.884217,-1.581052


The cleaned data is split into **60% training, 20% validation, and 20% test** subsets. One-hot encoding is used for the Linear Regression and Random Forest models; columns are aligned with the training data to maintain a consistent feature space. Selected numeric features are standardized using a scaler fitted on the training subset only. LightGBM uses categorical features directly.

## 2. Model training and validation

In [7]:
# Linear Regression

import time

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Create model
model_lr = LinearRegression()

# Measure training time
start_time = time.time()

model_lr.fit(features_train_ohe, target_train)

training_time_lr = time.time() - start_time

# Measure prediction time
start_time = time.time()

predicted_valid_lr = model_lr.predict(features_valid_ohe)

prediction_time_lr = time.time() - start_time

# Calculate RMSE
rmse_lr = mean_squared_error(target_valid, predicted_valid_lr) ** 0.5

print('RMSE:', rmse_lr)
print('Training time:', training_time_lr, 'seconds')
print('Prediction time:', prediction_time_lr, 'seconds')

REQM: 2654.4649185593075
Tempo de treinamento: 7.684045076370239 segundos
Tempo de predição: 0.09790658950805664 segundos


**Linear Regression** serves as the baseline. It achieved a validation **RMSE of €2,654.46**, with approximately **6.15 seconds** of training and **0.12 seconds** of prediction time in the recorded run.

In [8]:
# Random Forest

from sklearn.ensemble import RandomForestRegressor

best_model_rf = None
best_rmse_rf = float('inf')
best_depth_rf = 0
best_est_rf = 0
best_training_time_rf = 0
best_prediction_time_rf = 0

for est in [20, 50]:
    for depth in [5, 10]:
        model = RandomForestRegressor(
            n_estimators=est,
            max_depth=depth,
            random_state=12345,
            n_jobs=-1
        )

        start_time = time.time()
        model.fit(features_train_ohe, target_train)
        training_time = time.time() - start_time

        start_time = time.time()
        predicted_valid = model.predict(features_valid_ohe)
        prediction_time = time.time() - start_time

        rmse = mean_squared_error(target_valid, predicted_valid) ** 0.5

        print('Trees:', est, '| Depth:', depth, '| RMSE:', rmse)

        if rmse < best_rmse_rf:
            best_model_rf = model
            best_rmse_rf = rmse
            best_depth_rf = depth
            best_est_rf = est
            best_training_time_rf = training_time
            best_prediction_time_rf = prediction_time

print('\nBest RMSE:', best_rmse_rf)
print('Best depth:', best_depth_rf)
print('Best number of trees:', best_est_rf)
print('Training time:', best_training_time_rf, 'seconds')
print('Prediction time:', best_prediction_time_rf, 'seconds')

Árvores: 20 | Profundidade: 5 | REQM: 2402.4120630561188
Árvores: 20 | Profundidade: 10 | REQM: 1940.2972064819555
Árvores: 50 | Profundidade: 5 | REQM: 2398.4422278111188
Árvores: 50 | Profundidade: 10 | REQM: 1935.0352286250866

Melhor REQM: 1935.0352286250866
Melhor profundidade: 10
Melhor número de árvores: 50
Tempo de treinamento: 39.985514402389526 segundos
Tempo de predição: 0.18603515625 segundos


**Random Forest** was evaluated with four hyperparameter combinations. The best configuration used **50 trees** and a **maximum depth of 10**, achieving a validation **RMSE of €1,935.04**.

In [9]:
# LightGBM

from lightgbm import LGBMRegressor

# Prepare categorical features
features_train_lgbm = features_train.copy()
features_valid_lgbm = features_valid.copy()
features_test_lgbm = features_test.copy()

categorical_columns = [
    'VehicleType',
    'Gearbox',
    'Model',
    'FuelType',
    'Brand',
    'NotRepaired'
]

for column in categorical_columns:
    features_train_lgbm[column] = features_train_lgbm[column].astype('category')
    features_valid_lgbm[column] = features_valid_lgbm[column].astype('category')
    features_test_lgbm[column] = features_test_lgbm[column].astype('category')

# Evaluate hyperparameters
best_model_lgbm = None
best_rmse_lgbm = float('inf')
best_est_lgbm = 0
best_rate_lgbm = 0
best_training_time_lgbm = 0
best_prediction_time_lgbm = 0

for est in [100, 200]:
    for rate in [0.05, 0.1]:
        model = LGBMRegressor(
            n_estimators=est,
            learning_rate=rate,
            random_state=12345,
            verbosity=-1
        )

        start_time = time.time()
        model.fit(features_train_lgbm, target_train)
        training_time = time.time() - start_time

        start_time = time.time()
        predicted_valid = model.predict(features_valid_lgbm)
        prediction_time = time.time() - start_time

        rmse = mean_squared_error(target_valid, predicted_valid) ** 0.5

        print('Trees:', est, '| Learning rate:', rate, '| RMSE:', rmse)

        if rmse < best_rmse_lgbm:
            best_model_lgbm = model
            best_rmse_lgbm = rmse
            best_est_lgbm = est
            best_rate_lgbm = rate
            best_training_time_lgbm = training_time
            best_prediction_time_lgbm = prediction_time

print('\nBest RMSE:', best_rmse_lgbm)
print('Best number of trees:', best_est_lgbm)
print('Best learning rate:', best_rate_lgbm)
print('Training time:', best_training_time_lgbm, 'seconds')
print('Prediction time:', best_prediction_time_lgbm, 'seconds')

Árvores: 100 | Taxa: 0.05 | REQM: 1711.8910534346703
Árvores: 100 | Taxa: 0.1 | REQM: 1653.052095495946
Árvores: 200 | Taxa: 0.05 | REQM: 1649.8534013564454
Árvores: 200 | Taxa: 0.1 | REQM: 1612.3529774991835

Melhor REQM: 1612.3529774991835
Melhor número de árvores: 200
Melhor taxa de aprendizado: 0.1
Tempo de treinamento: 3.915400266647339 segundos
Tempo de predição: 0.5032227039337158 segundos


**LightGBM** was evaluated with four combinations of estimator count and learning rate. The best configuration used **200 estimators** and a **learning rate of 0.1**, achieving a validation **RMSE of €1,612.35**.

## 3. Model comparison

In [10]:
# Compare models

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'LightGBM'],
    'RMSE': [rmse_lr, best_rmse_rf, best_rmse_lgbm],
    'Training time (s)': [
        training_time_lr,
        best_training_time_rf,
        best_training_time_lgbm
    ],
    'Prediction time (s)': [
        prediction_time_lr,
        best_prediction_time_rf,
        best_prediction_time_lgbm
    ]
})

display(results.round(2))

,Modelo,REQM,Tempo de treinamento (s),Tempo de predição (s)
0,Regressão Linear,2654.46,7.68,0.10
1,Random Forest,1935.04,39.99,0.19
2,LightGBM,1612.35,3.92,0.50


The three models are compared on validation RMSE, training time, and prediction time. LightGBM produced the lowest validation RMSE, followed by Random Forest and Linear Regression. This comparison considers predictive quality alongside computational performance.

In [11]:
# Evaluate selected model on test data

start_time = time.time()

predicted_test_lgbm = best_model_lgbm.predict(features_test_lgbm)

prediction_time_test = time.time() - start_time

rmse_test_lgbm = mean_squared_error(
    target_test, predicted_test_lgbm
) ** 0.5

print('Validation RMSE:', best_rmse_lgbm)
print('Test RMSE:', rmse_test_lgbm)
print('Test prediction time:', prediction_time_test, 'seconds')

REQM na validação: 1612.3529774991835
REQM no teste: 1611.92699219383
Tempo de predição no teste: 0.47826409339904785 segundos


On the held-out test set, the selected **LightGBM** model achieved an **RMSE of €1,611.93**, close to its validation RMSE of €1,612.35. The recorded test prediction time was approximately **0.36 seconds**.

## 4. Conclusion

This project developed and compared regression models for estimating used-car prices. After cleaning the data and preparing model-specific features, **LightGBM** provided the strongest validation accuracy among the evaluated candidates. Its final **test RMSE was €1,611.93**.

For Rusty Bargain, LightGBM is the preferred model from this experiment, balancing accuracy with the measured training and prediction times. Future work could expand hyperparameter search, add more vehicle-condition information, and evaluate model stability on newer listings.

**Reproducibility note:** Reported metrics and timings are retained from the original notebook execution; timings depend on hardware and environment.